# Entrenamiento consolidado - Snake Species Pilot 51

Este notebook unifica lo que se uso en los notebooks anteriores para llegar al estado actual del proyecto.

Ruta final del proyecto:

```text
imagen -> especie -> riesgo venenosa/no venenosa -> decision segura
```

Modelo de produccion recomendado:

```text
B4 EfficientNetV2B1, imagen 260x260
```

Modelo experimental:

```text
B5 EfficientNetV2B3, imagen 300x300
```

B4 queda activo porque fue el modelo mas seguro para cliente. B5 mejora algunas metricas, pero no se promueve por falsos seguros en holdout completo.

## 1. Resumen de como se llego hasta aqui

Experimentos realizados:

```text
B0: EfficientNetV2B0 224x224. Primer modelo fuerte por especie.
B1: EfficientNetV2B1 260x260. Mejoro a B0.
B2: refuerzo dirigido de pares confundidos. No mejoro globalmente.
B3: refuerzo balanceado. Mejoro algunos casos, pero no reemplazo a B1.
B4: escala general del dataset. Mejor perfil de seguridad global.
B5: EfficientNetV2B3 300x300. Mejor exactitud, pero no mas seguro.
```

Este notebook deja el flujo limpio para entrenar B4 o B5 sin repetir celdas antiguas.

## 2. Montar Drive y escoger experimento

Ejecuta esta celda primero. Por defecto entrena B4, que es el modelo de produccion.

Si quieres repetir B5, cambia:

```python
EXPERIMENT = 'b5'
```

In [ ]:
# Importa Path para manejar rutas de forma clara.
from pathlib import Path

# Importa json para guardar class_names y metricas.
import json

# Importa shutil para copiar modelos ganadores y crear ZIPs.
import shutil

# Importa zipfile para extraer el dataset desde Google Drive.
import zipfile

# Monta Google Drive dentro de Colab.
from google.colab import drive

# Activa el acceso a los archivos de Drive.
drive.mount('/content/drive')

# Define el experimento que se va a ejecutar: 'b4' produccion o 'b5' candidato.
EXPERIMENT = 'b4'

# Define la carpeta base del proyecto dentro de Drive.
DRIVE_PROJECT_DIR = Path('/content/drive/MyDrive/PROYECTO')

# Define la carpeta temporal de trabajo dentro de Colab.
PROJECT_DIR = Path('/content/snake_project')

# Define donde se extraen los datasets dentro de Colab.
DATA_PARENT = PROJECT_DIR / 'data/active'

# Define la ruta del ZIP B4 Scale dentro de Drive.
ZIP_PATH = DRIVE_PROJECT_DIR / 'snake_species_pilot51_b4_scale.zip'

# Valida que el ZIP exista antes de seguir.
assert ZIP_PATH.exists(), f'No se encontro el ZIP: {ZIP_PATH}'

# Define configuraciones por experimento.
EXPERIMENTS = {
    'b4': {
        'dataset_name': 'Snake Species Pilot 51 Targeted B4 Scale',
        'output_name': 'species_pilot51_b4_efficientnetv2b1_colab',
        'architecture': 'EfficientNetV2B1',
        'image_size': (260, 260),
        'batch_size': 16,
        'learning_rate': 1e-4,
        'weight_decay': 1e-5,
        'epochs': 22,
    },
    'b5': {
        'dataset_name': 'Snake Species Pilot 51 Targeted B4 Scale',
        'output_name': 'species_pilot51_b5_efficientnetv2b3_300_colab',
        'architecture': 'EfficientNetV2B3',
        'image_size': (300, 300),
        'batch_size': 8,
        'learning_rate': 1e-4,
        'weight_decay': 1e-5,
        'epochs': 24,
    },
}

# Selecciona la configuracion del experimento activo.
CONFIG = EXPERIMENTS[EXPERIMENT]

# Define la carpeta final del dataset extraido.
DATA_DIR = DATA_PARENT / CONFIG['dataset_name']

# Define la carpeta donde se guardara el modelo entrenado.
OUTPUT_DIR = DRIVE_PROJECT_DIR / 'models' / CONFIG['output_name']

# Crea la carpeta de salida si no existe.
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Muestra configuracion para verificar antes de entrenar.
print('EXPERIMENT:', EXPERIMENT)
print('ZIP_PATH:', ZIP_PATH)
print('DATA_DIR:', DATA_DIR)
print('OUTPUT_DIR:', OUTPUT_DIR)
print('CONFIG:', CONFIG)

## 3. Extraer dataset

Esta celda borra solamente la copia temporal de Colab y vuelve a extraer el ZIP. No borra el ZIP de Drive.

In [ ]:
# Borra la copia temporal del dataset si ya existia en esta sesion de Colab.
if DATA_DIR.exists():
    shutil.rmtree(DATA_DIR)

# Crea la carpeta padre donde se va a extraer el dataset.
DATA_PARENT.mkdir(parents=True, exist_ok=True)

# Abre el ZIP del dataset.
with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
    # Extrae cada miembro normalizando separadores de Windows o Linux.
    for member in zip_ref.infolist():
        # Normaliza barras invertidas a barras normales.
        normalized_name = member.filename.replace('\\', '/')
        # Ignora entradas vacias.
        if not normalized_name.strip():
            continue
        # Construye la ruta destino dentro de data/active.
        target_path = DATA_PARENT / normalized_name
        # Si el miembro es carpeta, crea la carpeta.
        if normalized_name.endswith('/'):
            target_path.mkdir(parents=True, exist_ok=True)
            continue
        # Crea la carpeta padre del archivo.
        target_path.parent.mkdir(parents=True, exist_ok=True)
        # Copia el archivo desde el ZIP al destino.
        with zip_ref.open(member) as source, open(target_path, 'wb') as target:
            shutil.copyfileobj(source, target)

# Verifica que la carpeta principal exista.
assert DATA_DIR.exists(), f'No se encontro DATA_DIR: {DATA_DIR}'

# Verifica que existan los tres splits.
assert (DATA_DIR / 'train').exists(), 'Falta train/'
assert (DATA_DIR / 'val').exists(), 'Falta val/'
assert (DATA_DIR / 'test').exists(), 'Falta test/'

# Cuenta imagenes por split para confirmar extraccion.
for split in ['train', 'val', 'test']:
    count = sum(1 for path in (DATA_DIR / split).rglob('*') if path.is_file())
    print(split, count)

## 4. Preparar TensorFlow y GPU

Activa mixed precision si Colab entrega GPU. Esto acelera entrenamiento y reduce memoria.

In [ ]:
# Importa TensorFlow.
import tensorflow as tf

# Importa Keras desde TensorFlow.
from tensorflow import keras

# Muestra la version de TensorFlow.
print('TensorFlow:', tf.__version__)

# Lista GPUs disponibles.
gpus = tf.config.list_physical_devices('GPU')

# Muestra las GPUs encontradas.
print('GPU:', gpus)

# Si hay GPU, activa mixed precision.
if gpus:
    keras.mixed_precision.set_global_policy('mixed_float16')
    print('Mixed precision activa:', keras.mixed_precision.global_policy())

## 5. Cargar datasets

Se cargan `train`, `val` y `test` desde carpetas. La etiqueta es categorica porque tenemos 51 especies.

In [ ]:
# Define el tamano de imagen desde la configuracion.
IMG_SIZE = CONFIG['image_size']

# Define el batch size desde la configuracion.
BATCH_SIZE = CONFIG['batch_size']

# Define una semilla para reproducibilidad.
SEED = 42

# Carga el dataset de entrenamiento.
train_ds_raw = keras.utils.image_dataset_from_directory(
    DATA_DIR / 'train',
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='categorical',
    shuffle=True,
    seed=SEED,
)

# Carga el dataset de validacion.
val_ds_raw = keras.utils.image_dataset_from_directory(
    DATA_DIR / 'val',
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='categorical',
    shuffle=False,
)

# Carga el dataset de prueba.
test_ds_raw = keras.utils.image_dataset_from_directory(
    DATA_DIR / 'test',
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='categorical',
    shuffle=False,
)

# Guarda los nombres de clases en el mismo orden que Keras.
class_names = train_ds_raw.class_names

# Cuenta cuantas clases hay.
num_classes = len(class_names)

# Guarda class_names para que la app sepa interpretar la salida del modelo.
with open(OUTPUT_DIR / 'class_names.json', 'w', encoding='utf-8') as file:
    json.dump(class_names, file, indent=2)

# Activa prefetch para mejorar rendimiento.
AUTOTUNE = tf.data.AUTOTUNE

# Prefetch en train.
train_ds = train_ds_raw.prefetch(AUTOTUNE)

# Prefetch en validacion.
val_ds = val_ds_raw.prefetch(AUTOTUNE)

# Prefetch en test.
test_ds = test_ds_raw.prefetch(AUTOTUNE)

# Muestra resumen de clases.
print('Clases:', num_classes)
print('class_names guardado en:', OUTPUT_DIR / 'class_names.json')

## 6. Calcular pesos de clase

Los pesos ayudan cuando algunas especies tienen menos imagenes que otras.

In [ ]:
# Importa numpy para arreglos numericos.
import numpy as np

# Importa compute_class_weight para balancear clases.
from sklearn.utils.class_weight import compute_class_weight

# Crea una lista vacia de etiquetas.
labels = []

# Recorre cada clase con su indice.
for idx, class_name in enumerate(class_names):
    # Ubica la carpeta de esa clase en train.
    class_dir = DATA_DIR / 'train' / class_name
    # Cuenta archivos de imagen dentro de esa clase.
    image_count = len([path for path in class_dir.iterdir() if path.is_file()])
    # Agrega el indice tantas veces como imagenes existan.
    labels.extend([idx] * image_count)

# Calcula pesos balanceados por clase.
weights = compute_class_weight(
    class_weight='balanced',
    classes=np.arange(num_classes),
    y=np.array(labels),
)

# Convierte pesos a diccionario para model.fit.
class_weights = {idx: float(weight) for idx, weight in enumerate(weights)}

# Muestra diagnostico general.
print('Total etiquetas:', len(labels))
print('Pesos calculados:', len(class_weights))
print('Peso minimo:', min(class_weights.values()))
print('Peso maximo:', max(class_weights.values()))

## 7. Construir modelo

Se usa transferencia de aprendizaje. La base EfficientNet viene preentrenada en ImageNet y se agrega una cabeza para 51 especies.

In [ ]:
# Importa capas de Keras.
from tensorflow.keras import layers

# Escoge la arquitectura base segun la configuracion.
if CONFIG['architecture'] == 'EfficientNetV2B1':
    BaseModel = keras.applications.EfficientNetV2B1
elif CONFIG['architecture'] == 'EfficientNetV2B3':
    BaseModel = keras.applications.EfficientNetV2B3
else:
    raise ValueError(f"Arquitectura no soportada: {CONFIG['architecture']}")

# Define aumento de datos para mejorar generalizacion.
data_augmentation = keras.Sequential(
    [
        layers.RandomFlip('horizontal', seed=SEED),
        layers.RandomRotation(0.08, seed=SEED),
        layers.RandomZoom(0.12, seed=SEED),
        layers.RandomContrast(0.12, seed=SEED),
    ],
    name='data_augmentation',
)

# Carga la base EfficientNet sin la cabeza original.
base_model = BaseModel(
    include_top=False,
    weights='imagenet',
    input_shape=IMG_SIZE + (3,),
)

# Congela la base para entrenar primero solo la cabeza.
base_model.trainable = False

# Define la entrada del modelo.
inputs = keras.Input(shape=IMG_SIZE + (3,))

# Aplica aumento de datos a la entrada.
x = data_augmentation(inputs)

# Pasa la imagen por la base preentrenada.
x = base_model(x, training=False)

# Convierte mapas de activacion en vector.
x = layers.GlobalAveragePooling2D(name='avg_pool')(x)

# Agrega dropout para reducir sobreajuste.
x = layers.Dropout(0.35, name='dropout')(x)

# Crea la capa final softmax en float32 para estabilidad con mixed precision.
outputs = layers.Dense(num_classes, activation='softmax', dtype='float32', name='predictions')(x)

# Construye el modelo completo.
model = keras.Model(inputs, outputs, name=f"species_{CONFIG['architecture'].lower()}")

# Intenta usar AdamW con weight decay.
try:
    optimizer = keras.optimizers.AdamW(
        learning_rate=CONFIG['learning_rate'],
        weight_decay=CONFIG['weight_decay'],
    )
# Si la version no soporta AdamW, usa Adam.
except Exception:
    optimizer = keras.optimizers.Adam(learning_rate=CONFIG['learning_rate'])

# Compila el modelo con perdida categorica y metricas top-k.
model.compile(
    optimizer=optimizer,
    loss=keras.losses.CategoricalCrossentropy(label_smoothing=0.05),
    metrics=[
        keras.metrics.CategoricalAccuracy(name='accuracy'),
        keras.metrics.TopKCategoricalAccuracy(k=3, name='top3'),
        keras.metrics.TopKCategoricalAccuracy(k=5, name='top5'),
    ],
)

# Muestra formas de entrada y salida.
print('Modelo construido:', model.name)
print('Entrada:', model.input_shape)
print('Salida:', model.output_shape)
print('Capas base:', len(base_model.layers))

## 8. Entrenar cabeza

Esta fue la fase que dio el mejor resultado en B4 y B5. Se guarda el mejor modelo por `val_loss`.

In [ ]:
# Define callbacks para entrenamiento de cabeza.
head_callbacks = [
    keras.callbacks.ModelCheckpoint(
        OUTPUT_DIR / 'best_head_model.keras',
        monitor='val_loss',
        save_best_only=True,
        verbose=1,
    ),
    keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=5,
        restore_best_weights=True,
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.2,
        patience=2,
        min_lr=1e-7,
    ),
]

# Entrena la cabeza del modelo.
history_head = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=CONFIG['epochs'],
    callbacks=head_callbacks,
    class_weight=class_weights,
)

# Guarda tambien el modelo final de cabeza.
model.save(OUTPUT_DIR / 'final_head_model.keras')

# Muestra ubicacion del mejor modelo.
print('best_head_model:', OUTPUT_DIR / 'best_head_model.keras')
print('final_head_model:', OUTPUT_DIR / 'final_head_model.keras')

## 9. Evaluar modelo de cabeza

Se evalua en test. Esta metrica se usa para comparar contra experimentos anteriores.

In [ ]:
# Carga el mejor modelo guardado durante entrenamiento.
best_head_model = keras.models.load_model(OUTPUT_DIR / 'best_head_model.keras')

# Evalua el modelo en test.
head_test_metrics = best_head_model.evaluate(test_ds, return_dict=True)

# Muestra metricas de test.
print(head_test_metrics)

## 10. Reporte completo y matriz de confusion

Calcula accuracy, macro F1, weighted F1, top-3 y top-5. Tambien guarda matriz de confusion y reporte por clase.

In [ ]:
# Importa metricas de sklearn.
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score, top_k_accuracy_score

# Importa pandas para guardar tablas.
import pandas as pd

# Crea listas para etiquetas reales.
y_true = []

# Crea listas para etiquetas predichas.
y_pred = []

# Crea lista para probabilidades completas.
y_prob = []

# Recorre el dataset de test por lotes.
for x_batch, y_batch in test_ds:
    # Predice probabilidades del lote.
    probabilities = best_head_model.predict(x_batch, verbose=0)
    # Convierte one-hot real a indice.
    true_indices = np.argmax(y_batch.numpy(), axis=1)
    # Convierte probabilidades a clase top-1.
    pred_indices = np.argmax(probabilities, axis=1)
    # Acumula etiquetas reales.
    y_true.extend(true_indices.tolist())
    # Acumula predicciones.
    y_pred.extend(pred_indices.tolist())
    # Acumula probabilidades.
    y_prob.extend(probabilities.tolist())

# Convierte reales a numpy.
y_true = np.array(y_true)

# Convierte predicciones a numpy.
y_pred = np.array(y_pred)

# Convierte probabilidades a numpy.
y_prob = np.array(y_prob)

# Calcula matriz de confusion.
cm = confusion_matrix(y_true, y_pred)

# Calcula reporte por clase.
report = classification_report(y_true, y_pred, target_names=class_names, output_dict=True, zero_division=0)

# Calcula resumen de metricas.
summary = {
    'selected_model': 'best_head_model.keras',
    'accuracy': float(accuracy_score(y_true, y_pred)),
    'macro_f1': float(f1_score(y_true, y_pred, average='macro')),
    'weighted_f1': float(f1_score(y_true, y_pred, average='weighted')),
    'top3_accuracy': float(top_k_accuracy_score(y_true, y_prob, k=3, labels=np.arange(num_classes))),
    'top5_accuracy': float(top_k_accuracy_score(y_true, y_prob, k=5, labels=np.arange(num_classes))),
    'support': int(len(y_true)),
}

# Guarda resumen de evaluacion.
with open(OUTPUT_DIR / 'evaluation_summary.json', 'w', encoding='utf-8') as file:
    json.dump(summary, file, indent=2)

# Guarda reporte de clasificacion.
with open(OUTPUT_DIR / 'classification_report.json', 'w', encoding='utf-8') as file:
    json.dump(report, file, indent=2)

# Guarda matriz de confusion como JSON.
with open(OUTPUT_DIR / 'confusion_matrix.json', 'w', encoding='utf-8') as file:
    json.dump(cm.tolist(), file)

# Guarda resultados por imagen/clase en CSV simple.
pd.DataFrame({
    'y_true': y_true,
    'y_pred': y_pred,
    'true_class': [class_names[i] for i in y_true],
    'pred_class': [class_names[i] for i in y_pred],
}).to_csv(OUTPUT_DIR / 'test_predictions.csv', index=False)

# Muestra resumen final.
print(json.dumps(summary, indent=2))

## 11. Graficar matriz de confusion

La diagonal muestra aciertos. Puntos fuera de la diagonal muestran confusiones entre especies.

In [ ]:
# Importa matplotlib para graficar.
import matplotlib.pyplot as plt

# Importa seaborn para heatmap.
import seaborn as sns

# Define tamano de figura.
plt.figure(figsize=(18, 18))

# Dibuja matriz de confusion.
sns.heatmap(cm, cmap='Blues', xticklabels=False, yticklabels=False, square=True, cbar=True)

# Agrega titulo.
plt.title(f"Matriz de confusion - {CONFIG['output_name']}")

# Agrega etiqueta eje X.
plt.xlabel('Prediccion')

# Agrega etiqueta eje Y.
plt.ylabel('Real')

# Ajusta layout.
plt.tight_layout()

# Guarda imagen en OUTPUT_DIR.
plt.savefig(OUTPUT_DIR / 'confusion_matrix.png', dpi=180)

# Muestra grafica.
plt.show()

## 12. Promover modelo ganador

En B4 y B5 el modelo de cabeza fue el ganador practico. Se copia como `best_model.keras`, que es el nombre usado por las apps.

In [ ]:
# Define ruta del mejor modelo de cabeza.
best_head_path = OUTPUT_DIR / 'best_head_model.keras'

# Define ruta estandar para apps.
best_model_path = OUTPUT_DIR / 'best_model.keras'

# Copia el mejor modelo de cabeza como best_model.keras.
shutil.copy2(best_head_path, best_model_path)

# Guarda resumen de seleccion.
selection_summary = {
    'selected_model': 'best_head_model.keras',
    'deployed_as': 'best_model.keras',
    'reason': 'La cabeza entrenada fue la opcion estable; fine-tuning solo debe promoverse si mejora validacion y seguridad.',
    'experiment': EXPERIMENT,
    'config': CONFIG,
    'test_summary': summary,
}

# Escribe resumen de seleccion.
with open(OUTPUT_DIR / 'model_selection_summary.json', 'w', encoding='utf-8') as file:
    json.dump(selection_summary, file, indent=2)

# Muestra ruta final.
print('Modelo ganador:', best_model_path)

## 13. Fine-tuning opcional

Esta celda queda como opcional. En el proyecto, el fine-tuning no se promovio automaticamente porque en varios experimentos no supero al modelo de cabeza.

Ejecutala solo si quieres experimentar y compara contra `best_head_model.keras` antes de reemplazar.

In [ ]:
# Cambia a True solo si quieres ejecutar fine-tuning.
RUN_FINE_TUNING = False

# Ejecuta fine-tuning solo si la bandera esta activa.
if RUN_FINE_TUNING:
    # Descongela la base EfficientNet.
    base_model.trainable = True
    # Congela la mayor parte de la red y deja entrenable el tramo final.
    fine_tune_at = int(len(base_model.layers) * 0.80)
    # Recorre las capas anteriores al punto de fine-tuning.
    for layer in base_model.layers[:fine_tune_at]:
        # Mantiene congeladas esas capas.
        layer.trainable = False
    # Intenta usar AdamW con learning rate muy bajo.
    try:
        fine_optimizer = keras.optimizers.AdamW(learning_rate=1e-6, weight_decay=1e-6)
    # Si falla AdamW, usa Adam.
    except Exception:
        fine_optimizer = keras.optimizers.Adam(learning_rate=1e-6)
    # Recompila el modelo para aplicar capas entrenables nuevas.
    model.compile(
        optimizer=fine_optimizer,
        loss=keras.losses.CategoricalCrossentropy(label_smoothing=0.05),
        metrics=[
            keras.metrics.CategoricalAccuracy(name='accuracy'),
            keras.metrics.TopKCategoricalAccuracy(k=3, name='top3'),
            keras.metrics.TopKCategoricalAccuracy(k=5, name='top5'),
        ],
    )
    # Define callbacks de fine-tuning.
    fine_callbacks = [
        keras.callbacks.ModelCheckpoint(OUTPUT_DIR / 'best_finetuned_model.keras', monitor='val_loss', save_best_only=True, verbose=1),
        keras.callbacks.EarlyStopping(monitor='val_loss', patience=4, restore_best_weights=True),
        keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=2, min_lr=1e-7),
    ]
    # Entrena pocas epocas con learning rate bajo.
    history_fine = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=10,
        callbacks=fine_callbacks,
        class_weight=class_weights,
    )
    # Guarda modelo final fine-tuned.
    model.save(OUTPUT_DIR / 'final_finetuned_model.keras')
    # Muestra mensaje de cierre.
    print('Fine-tuning terminado. Compara antes de promover.')
else:
    # Informa que no se ejecuto fine-tuning.
    print('Fine-tuning omitido. Esta es la opcion recomendada para reproducir B4/B5 estable.')

## 14. Comprimir modelo para descargar o subir al proyecto local

Esta celda crea un ZIP con modelo, clases, metricas y matriz de confusion.

In [ ]:
# Define ruta del ZIP final dentro de la carpeta models de Drive.
ZIP_MODEL_PATH = OUTPUT_DIR.parent / f"{CONFIG['output_name']}.zip"

# Borra un ZIP anterior si ya existia.
if ZIP_MODEL_PATH.exists():
    ZIP_MODEL_PATH.unlink()

# Crea ZIP desde toda la carpeta OUTPUT_DIR.
shutil.make_archive(str(ZIP_MODEL_PATH).replace('.zip', ''), 'zip', OUTPUT_DIR)

# Muestra ruta final del ZIP.
print('ZIP creado:', ZIP_MODEL_PATH)

## 15. Criterio final de seleccion

Para cliente se eligio B4.

Criterio:

```text
1. No basta con mayor accuracy.
2. El error mas grave es falso seguro: venenosa -> no venenosa.
3. B4 mantuvo mejor seguridad global.
4. B5 queda como candidato experimental.
```

La app debe usar `best_model.keras` de B4 y conservar la politica `B4 safety-first`.